In [15]:
import pandas as pd
import glob
import json

In [16]:
# 1. Definir rutas (Ajusta a tus carpetas reales)
ruta_incendios = r'C:\Users\etham\DataspellProjects\Modelo-Machine-Learning-Para-Detectar-Incendios\Incendios\*.csv'
ruta_no_incendios = r'C:\Users\etham\DataspellProjects\Modelo-Machine-Learning-Para-Detectar-Incendios\NoIncendios\*.csv'

In [17]:
# 2. Procesar dataset de INCENDIOS
archivos_incendios = glob.glob(ruta_incendios)
lista_df_incendios = []

for archivo in archivos_incendios:
    df = pd.read_csv(archivo)
    # Seleccionamos solo lo que nos sirve para predecir
    df_limpio = df[['latitude', 'longitude', 'acq_date']].copy()
    df_limpio['target'] = 1  # Etiqueta de incendio
    lista_df_incendios.append(df_limpio)

In [18]:
# 3. Procesar dataset de NO INCENDIOS
archivos_no_incendios = glob.glob(ruta_no_incendios)
lista_df_no_incendios = []

for archivo in archivos_no_incendios:
    df = pd.read_csv(archivo)

    # Extraer latitud y longitud de la columna '.geo' que viene en formato JSON string
    # En GeoJSON el orden siempre es [Longitud, Latitud]
    df['longitude'] = df['.geo'].apply(lambda x: json.loads(x)['coordinates'][0])
    df['latitude'] = df['.geo'].apply(lambda x: json.loads(x)['coordinates'][1])

    # Seleccionamos las mismas columnas para que haga "match" con el otro dataset
    df_limpio = df[['latitude', 'longitude', 'acq_date']].copy()
    df_limpio['target'] = 0  # Etiqueta de no incendio
    lista_df_no_incendios.append(df_limpio)

In [19]:
# 4. Concatenar todo en el Dataset Maestro
df_master = pd.concat(lista_df_incendios + lista_df_no_incendios, ignore_index=True)

In [20]:
# 5. Formatear la fecha y extraer variables temporales útiles para el modelo
df_master['Date'] = pd.to_datetime(df_master['acq_date'])
df_master['year'] = df_master['Date'].dt.year
df_master['month'] = df_master['Date'].dt.month
df_master['day'] = df_master['Date'].dt.day
df_master['day_of_year'] = df_master['Date'].dt.dayofyear

# Limpiar columnas temporales redundantes
df_master = df_master.drop(columns=['acq_date'])

print("¡Dataset Unificado Exitosamente!")
print(f"Total de registros: {len(df_master)}")
print("\nDistribución de clases:")
print(df_master['target'].value_counts())

# Comprobación visual de que todo cuadra
print("\nMuestra de los datos:")
print(df_master.sample(5))

¡Dataset Unificado Exitosamente!
Total de registros: 395403

Distribución de clases:
target
1    350089
0     45314
Name: count, dtype: int64

Muestra de los datos:
        latitude  longitude  target       Date  year  month  day  day_of_year
166394   -6.6629   -76.6772       1 2013-08-23  2013      8   23          235
136170   -8.1640   -75.0369       1 2011-08-12  2011      8   12          224
111281   -8.9303   -74.7860       1 2009-10-05  2009     10    5          278
109928   -8.9482   -75.0794       1 2009-09-17  2009      9   17          260
173020  -12.8831   -69.7214       1 2013-09-30  2013      9   30          273


In [21]:
# 1. Definir la caja delimitadora aproximada de la región San Martín
lat_min, lat_max = -8.8, -5.3
lon_min, lon_max = -77.8, -75.5

# 2. Filtrar el Dataset Maestro
df_sm = df_master[
    (df_master['latitude'] >= lat_min) & (df_master['latitude'] <= lat_max) &
    (df_master['longitude'] >= lon_min) & (df_master['longitude'] <= lon_max)
].copy()

print("¡Filtro Geográfico Aplicado!")
print(f"Total de registros en San Martín: {len(df_sm)}")
print("\nNueva distribución de clases:")
print(df_sm['target'].value_counts())

¡Filtro Geográfico Aplicado!
Total de registros en San Martín: 120597

Nueva distribución de clases:
target
1    77154
0    43443
Name: count, dtype: int64


In [22]:
# Separar las clases
df_incendios_sm = df_sm[df_sm['target'] == 1]
df_no_incendios_sm = df_sm[df_sm['target'] == 0]

# Encontrar cuál es la clase minoritaria
min_size = min(len(df_incendios_sm), len(df_no_incendios_sm))

# Muestrear aleatoriamente la clase mayoritaria para igualar a la minoritaria
df_incendios_bal = df_incendios_sm.sample(n=min_size, random_state=42)
df_no_incendios_bal = df_no_incendios_sm.sample(n=min_size, random_state=42)

# Unir y barajar (shuffle)
df_final = pd.concat([df_incendios_bal, df_no_incendios_bal]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset Final Balanceado: {len(df_final)} registros.")
print(df_final['target'].value_counts())

Dataset Final Balanceado: 86886 registros.
target
0    43443
1    43443
Name: count, dtype: int64


In [23]:
# Preparar columnas exactas para AppEEARS
df_appeears = pd.DataFrame()
df_appeears['id'] = range(1, len(df_final) + 1)
df_appeears['category'] = df_final['target'].apply(lambda x: 'incendio' if x == 1 else 'no_incendio')
df_appeears['latitude'] = df_final['latitude']
df_appeears['longitude'] = df_final['longitude']
df_appeears['year'] = df_final['year']

# Generar los 5 lotes
rangos = [(2000, 2004), (2005, 2009), (2010, 2014), (2015, 2019), (2020, 2024)]

for inicio, fin in rangos:
    filtro = df_appeears[(df_appeears['year'] >= inicio) & (df_appeears['year'] <= fin)]
    lote_final = filtro.drop(columns=['year']) # Quitamos 'year' para que AppEEARS no falle

    nombre_archivo = f'Coordenadas_SanMartin_{inicio}_{fin}.csv'
    lote_final.to_csv(nombre_archivo, index=False)
    print(f"Lote generado: {nombre_archivo} con {len(lote_final)} puntos.")

Lote generado: Coordenadas_SanMartin_2000_2004.csv con 14415 puntos.
Lote generado: Coordenadas_SanMartin_2005_2009.csv con 19743 puntos.
Lote generado: Coordenadas_SanMartin_2010_2014.csv con 18831 puntos.
Lote generado: Coordenadas_SanMartin_2015_2019.csv con 16699 puntos.
Lote generado: Coordenadas_SanMartin_2020_2024.csv con 17198 puntos.


In [24]:
import pandas as pd
import numpy as np

In [25]:
# 1. Preparar el DataFrame maestro (Asegúrate de que df_final esté ordenado por fecha)
df_final = df_final.sort_values('Date')

# --- CREACIÓN DE VARIABLES GEOGRÁFICAS Y TEMPORALES ---
# Distancia al ecuador (Aprox 111 km por cada grado de latitud)
df_final['Distance_to_equator'] = df_final['latitude'].abs() * 111

# Época seca (1 si es junio, julio, agosto o septiembre; 0 si no)
df_final['Is_dry_season'] = df_final['month'].isin([6, 7, 8, 9]).astype(int)

In [26]:
# 2. Cargar y limpiar Temperatura (MOD11A1) y NDVI (MOD13Q1)
import glob
archivos_temp = glob.glob('MOD11A1/*.csv')
lista_temp = []
for f in archivos_temp:
    df_t = pd.read_csv(f)
    df_t = df_t[['Latitude', 'Longitude', 'Date', 'MOD11A1_061_LST_Day_1km']].copy()
    lista_temp.append(df_t)
df_temp = pd.concat(lista_temp, ignore_index=True)
df_temp['Date'] = pd.to_datetime(df_temp['Date'])
df_temp['Temp_max'] = (df_temp['MOD11A1_061_LST_Day_1km'] * 0.02) - 273.15
df_temp = df_temp.dropna(subset=['Temp_max'])
df_temp = df_temp.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})
df_temp = df_temp.sort_values('Date')

archivos_ndvi = glob.glob('MOD13Q1/*.csv')
lista_ndvi = []
for f in archivos_ndvi:
    df_n = pd.read_csv(f)
    df_n = df_n[['Latitude', 'Longitude', 'Date', 'MOD13Q1_061__250m_16_days_NDVI']].copy()
    lista_ndvi.append(df_n)
df_ndvi = pd.concat(lista_ndvi, ignore_index=True)
df_ndvi['Date'] = pd.to_datetime(df_ndvi['Date'])
df_ndvi['NDVI'] = df_ndvi['MOD13Q1_061__250m_16_days_NDVI']
df_ndvi = df_ndvi.dropna(subset=['NDVI'])
df_ndvi = df_ndvi.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})
df_ndvi = df_ndvi.sort_values('Date')

# 3. Redondear coordenadas para poder hacer merge
df_final['lat_round'] = df_final['latitude'].round(2)
df_final['lon_round'] = df_final['longitude'].round(2)
df_temp['lat_round'] = df_temp['latitude'].round(2)
df_temp['lon_round'] = df_temp['longitude'].round(2)
df_ndvi['lat_round'] = df_ndvi['latitude'].round(2)
df_ndvi['lon_round'] = df_ndvi['longitude'].round(2)

# Merge con tolerancias para las fechas (asof merge)
df_final = df_final.sort_values('Date')
df_temp = df_temp.sort_values('Date')
df_ndvi = df_ndvi.sort_values('Date')

df_final = pd.merge_asof(df_final, df_temp[['Date', 'lat_round', 'lon_round', 'Temp_max']], on='Date', by=['lat_round', 'lon_round'], direction='nearest')
df_final = pd.merge_asof(df_final, df_ndvi[['Date', 'lat_round', 'lon_round', 'NDVI']], on='Date', by=['lat_round', 'lon_round'], direction='nearest')

# Llenar nulos si hay (opcional)
df_final['Temp_max'] = df_final['Temp_max'].fillna(df_final['Temp_max'].mean())
df_final['NDVI'] = df_final['NDVI'].fillna(df_final['NDVI'].mean())

df_final.to_csv('dataset_maestro_completo.csv', index=False)
print('Merge exitoso. Archivo dataset_maestro_completo.csv guardado.')


Merge exitoso. Archivo dataset_maestro_completo.csv guardado.
